# Information Retrieval Assignment - Combined: K-Means (BM25) + Rocchio-SMART (TF-IDF)

This notebook combines **Option 1** (K-Means clustering on document vectors with **BM25** weighting) and **Option 3** (Rocchio-**SMART** query refinement with **TF-IDF**). Together they cover:

- **Unsupervised structure**: grouping documents in vector space (clustering prototypes ~ "learned" themes).
- **Supervised / feedback-style retrieval**: moving a query vector toward relevant documents and away from non-relevant ones.

That pairing is especially relevant for **ML engineering roles** touching search, recommendations, and text representation: you use the same bag-of-words pipeline with different scorers (BM25 vs. TF-IDF) and both **centroid** ideas (cluster centers vs. Rocchio means).

**Submission note:** Run all cells so outputs (tables, printed terms) are visible before exporting or sharing (GitHub, Colab, etc.).


## Environment

Install dependencies if needed (e.g. first run on Colab or a fresh venv).


In [4]:
# Uncomment if packages are missing:
%pip install -q requests numpy pandas scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
import warnings
from collections import Counter

import numpy as np
from IPython.display import display
import pandas as pd
import requests
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

WIKI_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"


def tokenize(text: str) -> list[str]:
    text = text.lower()
    return re.findall(r"[a-z]{2,}", text)


def fetch_wikipedia_summary(title: str, timeout: float = 15.0) -> str:
    """Return plain-text extract from Wikipedia REST API (summary endpoint)."""
    url = WIKI_SUMMARY.format(title=title.replace(" ", "_"))
    headers = {"User-Agent": "IR-Assignment/1.0 (educational; classroom)"}
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    extract = data.get("extract") or ""
    return extract


def load_corpus(titles: list[str], fallback: dict[str, str] | None = None) -> tuple[list[str], list[str]]:
    """Load documents; use `fallback[title]` if the network request fails."""
    fallback = fallback or {}
    docs, labels = [], []
    for title in titles:
        try:
            text = fetch_wikipedia_summary(title)
            if len(text.strip()) < 80:
                raise ValueError("short extract")
        except Exception:
            text = fallback.get(title, "")
        docs.append(text)
        labels.append(title)
    return docs, labels

---

## Part 1 - Option 1: K-Means on BM25 document vectors (3 sports themes, K=3)

**Steps (per assignment):**

1. **Input:** ~20 short documents from three sports areas (Wikipedia summaries).
2. **Representation:** Bag-of-words with **BM25** weights per term-document pair, assembled into a sparse document-term matrix.
3. **Clustering:** **K-Means** with K=3. The assignment asks for **cosine** proximity. For **L2-normalized** rows, minimizing Euclidean distance to centroids is related to **cosine** distance (for unit vectors, ||x-y||^2 = 2 - 2*cos(theta)). We therefore **normalize** BM25 rows before `KMeans`.
4. **Display:** For each cluster prototype (centroid), show the **10 terms with largest weight** (we report terms that best align with the centroid direction).


In [6]:
def bm25_matrix(token_lists: list[list[str]], vocabulary: list[str]) -> sparse.csr_matrix:
    """BM25 Okapi BoW: rows = documents, columns = vocabulary order."""
    vocab_index = {w: i for i, w in enumerate(vocabulary)}
    n_docs = len(token_lists)
    dl = np.array([len(toks) for toks in token_lists], dtype=np.float64)
    avgdl = float(dl.mean()) if n_docs else 0.0
    k1, b = 1.5, 0.75

    df = np.zeros(len(vocabulary), dtype=np.int32)
    for toks in token_lists:
        seen = set(toks)
        for w in seen:
            if w in vocab_index:
                df[vocab_index[w]] += 1

    # BM25 idf (Robertson-Walker variant, +1 smoothing in log)
    idf = np.log((n_docs - df + 0.5) / (df + 0.5) + 1.0)

    rows, cols, data = [], [], []
    for i, toks in enumerate(token_lists):
        tf = Counter(toks)
        for term, f in tf.items():
            if term not in vocab_index:
                continue
            j = vocab_index[term]
            ff = float(f)
            denom = ff + k1 * (1.0 - b + b * dl[i] / avgdl)
            w = idf[j] * ((ff * (k1 + 1.0)) / denom)
            if w > 0:
                rows.append(i)
                cols.append(j)
                data.append(w)
    return sparse.csr_matrix((data, (rows, cols)), shape=(n_docs, len(vocabulary)))


def top_terms_for_centroid(
    centroid: np.ndarray, feature_names: np.ndarray, topn: int = 10
) -> pd.DataFrame:
    """Indices of largest absolute components (direction of prototype in term space)."""
    idx = np.argsort(np.abs(centroid))[-topn:][::-1]
    return pd.DataFrame(
        {
            "term": feature_names[idx],
            "centroid_weight": centroid[idx],
        }
    )


# --- Sports corpus (~20 articles, 3 fields) ---
SPORTS_TITLES = [
    # Association football
    "Association football",
    "FIFA World Cup",
    "Premier League",
    "UEFA Champions League",
    "Penalty kick (association football)",
    "Offside (association football)",
    "Football (word)",
    # Basketball
    "Basketball",
    "National Basketball Association",
    "Michael Jordan",
    "Three-point field goal",
    "Slam dunk",
    "Basketball positions",
    "EuroLeague",
    # Tennis
    "Tennis",
    "Grand Slam (tennis)",
    "Wimbledon Championships",
    "Serve (tennis)",
    "Tennis court",
    "Racket (sports equipment)",
]

SPORTS_FALLBACK = {
    "Association football": "Association football is a team sport played between two teams of eleven players with a spherical ball. It is played by millions worldwide and governed by FIFA.",
    "FIFA World Cup": "The FIFA World Cup is the international association football competition contested by national teams. It is held every four years.",
    "Premier League": "The Premier League is the top tier of English football featuring professional clubs in a seasonal competition.",
    "UEFA Champions League": "The UEFA Champions League is an annual club football competition organized by UEFA featuring top European teams.",
    "Penalty kick (association football)": "A penalty kick is a method of restarting play in football in which a player takes a free shot from the penalty mark.",
    "Offside (association football)": "Offside is a law in football stating that a player is offside if nearer to the opponents goal line than both the ball and the second last opponent.",
    "Football (word)": "The word football may mean different codes; in many countries it refers to association football.",
    "Basketball": "Basketball is a team sport in which two teams try to score by shooting a ball through a hoop.",
    "National Basketball Association": "The NBA is the premier professional basketball league in North America.",
    "Michael Jordan": "Michael Jordan is a former professional basketball player widely regarded as one of the greatest players.",
    "Three-point field goal": "A three point field goal is a basket made from beyond the three point arc in basketball.",
    "Slam dunk": "A slam dunk is a basketball shot performed by jumping and scoring by putting the ball through the rim.",
    "Basketball positions": "Basketball teams use positions such as point guard shooting guard forward and center.",
    "EuroLeague": "The EuroLeague is the top tier European professional basketball club competition.",
    "Tennis": "Tennis is a racket sport played individually or in doubles on a rectangular court separated by a net.",
    "Grand Slam (tennis)": "The Grand Slam tournaments are the four most important annual events in professional tennis.",
    "Wimbledon Championships": "Wimbledon is the oldest tennis tournament played on grass courts in London.",
    "Serve (tennis)": "The serve begins each point in tennis and must land in the diagonal service box.",
    "Tennis court": "A tennis court is the playing surface defined by lines for singles and doubles play.",
    "Racket (sports equipment)": "A racket is sports equipment consisting of a handled frame with an open hoop strung with cord.",
}

sports_docs_raw, sports_titles = load_corpus(SPORTS_TITLES, SPORTS_FALLBACK)
sports_token_lists = [tokenize(d) for d in sports_docs_raw]

# Build vocabulary from corpus (min_df trims noise; cap keeps matrix manageable)
min_df = 2
vec = CountVectorizer(
    preprocessor=lambda x: x,
    tokenizer=lambda x: tokenize(x),
    lowercase=False,
    token_pattern=None,
    min_df=min_df,
    max_df=0.95,
)
vec.fit(sports_docs_raw)
sports_vocab = vec.get_feature_names_out().tolist()
X_bm25 = bm25_matrix(sports_token_lists, sports_vocab)
print("Sports documents:", X_bm25.shape[0], "| BM25 vocabulary:", X_bm25.shape[1])

# L2-normalize rows: K-Means distance relates to cosine geometry on the sphere
X_bm25_n = normalize(X_bm25, norm="l2", axis=1)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_bm25_n)

# Centroids live in normalized space; map back to sparse term space direction for naming:
centroids = kmeans.cluster_centers_
fn = np.array(sports_vocab)

print("\nCluster sizes:", Counter(cluster_labels))

# Which documents fell in each cluster (for interpretation in the write-up)
sports_field = (
    ["football"] * 7 + ["basketball"] * 7 + ["tennis"] * 6
)
clust_df = pd.DataFrame(
    {"article": sports_titles, "sport_tag": sports_field, "cluster": cluster_labels}
).sort_values(["cluster", "sport_tag"])
print("\nDocuments by cluster (Wikipedia titles):")
display(clust_df)

for c in range(3):
    print(f"\n=== Cluster {c} - 10 strongest prototype terms (centroid coordinates) ===")
    display(top_terms_for_centroid(centroids[c], fn, 10))

Sports documents: 20 | BM25 vocabulary: 162

Cluster sizes: Counter({np.int32(0): 12, np.int32(1): 5, np.int32(2): 3})

Documents by cluster (Wikipedia titles):


,article,sport_tag,cluster
7,Basketball,basketball,0
9,Michael Jordan,basketball,0
10,Three-point field goal,basketball,0
11,Slam dunk,basketball,0
12,Basketball positions,basketball,0
0,Association football,football,0
4,Penalty kick (association football),football,0
5,Offside (association football),football,0
14,Tennis,tennis,0
17,Serve (tennis),tennis,0



=== Cluster 0 - 10 strongest prototype terms (centroid coordinates) ===


,term,centroid_weight
0,ball,0.090962
1,player,0.076114
2,three,0.072604
3,two,0.072560
4,shot,0.069373
5,goal,0.068522
6,court,0.067463
7,players,0.065778
8,play,0.064869
9,between,0.063728



=== Cluster 1 - 10 strongest prototype terms (centroid coordinates) ===


,term,centroid_weight
0,league,0.249747
1,professional,0.141624
2,most,0.131174
3,football,0.125500
4,european,0.118632
5,english,0.117293
6,sports,0.114574
7,premier,0.110231
8,national,0.104496
9,competition,0.103982



=== Cluster 2 - 10 strongest prototype terms (centroid coordinates) ===


,term,centroid_weight
0,tournament,0.217340
1,four,0.216077
2,grand,0.190001
3,year,0.180680
4,slam,0.159962
5,held,0.152746
6,all,0.139747
7,championships,0.139747
8,world,0.135045
9,tennis,0.131097


---

## Part 2 - Option 3: Rocchio-SMART with TF-IDF (alpha, beta, gamma)

**Setup (per assignment):**

1. **~50 documents** on one theme (**economics**), BoW with **TF-IDF**.
2. **Vocabulary pruning:** compute how common each term is across the corpus, **drop the 200 most frequent** terms (often stopword-like or overly generic), then take the **next 100** terms as the fixed vocabulary (the PDF wording "explosively" is treated as *explicitly* specifying that slice).
3. **Initial query q0:** three **content** words from that vocabulary.
4. **Rocchio-SMART** (standard textbook form: beta on relevant centroid, gamma on non-relevant centroid):

    q_opt = alpha * q0 + beta * mu_R - gamma * mu_NR

   with **alpha=0.5, beta=0.75, gamma=0.25**.

   If your course handout uses a different grouping for the Rocchio formula, match the lecture notes and re-run.

5. **Relevant vs. non-relevant:** random **20% / 80%** split (same idea as Option 2) so mu_R and mu_NR are centroids in TF-IDF space.

6. **Display (Part D):** top **five** features in q_opt and how their scores **compare** to the weights of the three q0 words (same TF-IDF coordinate system).


In [7]:
# ~50 economics-related Wikipedia pages
ECON_TITLES = [
    "Economics",
    "Microeconomics",
    "Macroeconomics",
    "Supply and demand",
    "Inflation",
    "Gross domestic product",
    "Monetary policy",
    "Fiscal policy",
    "Central bank",
    "Interest rate",
    "Exchange rate",
    "Market (economics)",
    "Perfect competition",
    "Monopoly",
    "Oligopoly",
    "Game theory",
    "Behavioral economics",
    "Econometrics",
    "Economic growth",
    "Recession",
    "Unemployment",
    "Consumer price index",
    "Deflation",
    "Quantitative easing",
    "Stock market",
    "Bond (finance)",
    "Derivative (finance)",
    "Comparative advantage",
    "Opportunity cost",
    "Externality",
    "Public good (economics)",
    "Tax",
    "Subsidy",
    "International trade",
    "World Trade Organization",
    "Economic inequality",
    "Poverty",
    "Human Development Index",
    "Economy of the United States",
    "European Central Bank",
    "Federal Reserve",
    "Adam Smith",
    "John Maynard Keynes",
    "Utility",
    "Marginalism",
    "Production–possibility frontier",
    "Capital (economics)",
    "Labour economics",
    "Rational choice theory",
    "Efficient-market hypothesis",
    "Financial crisis",
]

ECON_FALLBACK = {
    "Economics": "Economics is the social science that studies the production distribution and consumption of goods and services.",
    "Inflation": "Inflation is a sustained increase in the general price level of goods and services in an economy.",
    "Central bank": "A central bank manages a nations money supply credit and interest rates often targeting inflation or employment.",
    "Market (economics)": "In economics a market coordinates buyers and sellers for goods services or assets through prices.",
    "Stock market": "A stock market aggregates buyers and sellers of stocks which represent ownership claims in businesses.",
}

econ_docs_raw, econ_titles = load_corpus(ECON_TITLES, ECON_FALLBACK)
econ_token_lists = [tokenize(d) for d in econ_docs_raw]
print("Economics documents loaded:", len(econ_docs_raw))

# --- Vocabulary: drop 200 most frequent terms (by document frequency), keep next 100 ---
cv_full = CountVectorizer(
    preprocessor=lambda x: x,
    tokenizer=tokenize,
    lowercase=False,
    token_pattern=None,
    min_df=1,
)
X_bin = cv_full.fit_transform(econ_docs_raw)
dfs = np.asarray((X_bin > 0).sum(axis=0)).ravel()
terms_all = cv_full.get_feature_names_out()
order = np.argsort(-dfs)
sorted_terms = terms_all[order]
sorted_dfs = dfs[order]

if len(sorted_terms) < 300:
    raise ValueError(
        f"Need at least 300 distinct frequency-ranked terms; got {len(sorted_terms)}. "
        "Try lowering min_df or adding more pages."
    )

vocab_100 = sorted_terms[200:300].tolist()
print("Vocabulary slice: skipped top-200 by df; using terms ranked 201-300 (100 terms).")
print("Example skipped (very common):", list(sorted_terms[:5]))
print("Example kept vocab:", vocab_100[:8], "...")

# TF-IDF in the constrained 100-term space (smooth idf, L2 row norm is optional; we use default)
tfidf = TfidfVectorizer(
    preprocessor=lambda x: x,
    tokenizer=tokenize,
    lowercase=False,
    token_pattern=None,
    vocabulary=vocab_100,
    norm="l2",
    sublinear_tf=True,
    smooth_idf=True,
)
X_tfidf = tfidf.fit_transform(econ_docs_raw).tocsr()
feature_names = np.array(tfidf.get_feature_names_out())

# q0: three content words taken from the vocabulary slice itself. The slice is
# ordered by corpus document frequency (rank 201 = vocab_100[0], …), so we pick
# three well-separated indices as a transparent, reproducible choice. If a
# preferred economics term appears in this slice, we use it first.
preferred_q0 = [
    "econometrics",
    "oligopoly",
    "recession",
    "externality",
    "unemployment",
    "subsidy",
    "tariff",
    "macroeconomic",
]
q0_words = [w for w in preferred_q0 if w in tfidf.vocabulary_][:3]
slots = [5, 40, 80]
for pos in slots:
    if len(q0_words) >= 3:
        break
    if pos < len(vocab_100):
        w = vocab_100[pos]
        if w not in q0_words:
            q0_words.append(w)
while len(q0_words) < 3:
    q0_words.append(vocab_100[len(q0_words)])

print("Chosen q0 words (from the 100-term vocabulary):", q0_words)

q0 = np.zeros(len(feature_names))
for w in q0_words:
    q0[tfidf.vocabulary_[w]] = 1.0
q0 = normalize(q0.reshape(1, -1), norm="l2", axis=1).ravel()

# Random 20% relevant / 80% non-relevant (Option 2-style labeling)
n = X_tfidf.shape[0]
idx = np.arange(n)
rng = np.random.default_rng(42)
rng.shuffle(idx)
n_rel = max(1, int(round(0.2 * n)))
rel_idx = np.sort(idx[:n_rel])
nr_idx = np.sort(idx[n_rel:])

mu_R = np.asarray(X_tfidf[rel_idx].mean(axis=0)).ravel()
mu_NR = np.asarray(X_tfidf[nr_idx].mean(axis=0)).ravel()

alpha, beta, gamma = 0.5, 0.75, 0.25
q_opt = alpha * q0 + beta * mu_R - gamma * mu_NR

# --- Displays ---

def top_k(vec: np.ndarray, names: np.ndarray, k: int = 5) -> pd.DataFrame:
    idx = np.argsort(np.abs(vec))[-k:][::-1]
    return pd.DataFrame({"term": names[idx], "weight": vec[idx]})


print("\n--- Five strongest features in mu_R (relevant centroid) ---")
display(top_k(mu_R, feature_names, 5))

print("\n--- Five strongest features in mu_NR (non-relevant centroid) ---")
display(top_k(mu_NR, feature_names, 5))

print("\n--- Five strongest features in q_opt ---")
top5 = top_k(q_opt, feature_names, 5)
display(top5)

# Part D: top q_opt features + how those coordinates compare to q0 (same 100-dim space)
compare_rows = []
for _, row in top5.iterrows():
    term = str(row["term"])
    ti = int(np.where(feature_names == term)[0][0])
    row_dict = {
        "term": term,
        "q_opt": float(row["weight"]),
        "q0_at_same_coord": float(q0[ti]),
    }
    compare_rows.append(row_dict)

compare_df = pd.DataFrame(compare_rows)
print("\n--- Part D: q_opt top-5 - weight vs. q0 on that same coordinate ---")
display(compare_df)

q0_summary = pd.DataFrame(
    {
        "q0_word": q0_words,
        "q0_weight": [float(q0[tfidf.vocabulary_[w]]) for w in q0_words],
        "q_opt_weight": [float(q_opt[tfidf.vocabulary_[w]]) for w in q0_words],
    }
)
print("\nThe three q0 words: q0 vs. q_opt on those coordinates")
display(q0_summary)

# Nearest documents to q_opt (helps relate vector to actual pages)
sims = cosine_similarity(q_opt.reshape(1, -1), X_tfidf).ravel()
nearest = np.argsort(-sims)[:3]
print("\nThree documents closest to q_opt (cosine similarity):")
for rank, i in enumerate(nearest, 1):
    print(f"  {rank}. sim={sims[i]:.4f} - {econ_titles[i]}")

Economics documents loaded: 51
Vocabulary slice: skipped top-200 by df; using terms ranked 201-300 (100 terms).
Example skipped (very common): ['the', 'is', 'of', 'and', 'in']
Example kept vocab: ['sum', 'natural', 'direct', 'distribution', 'macroeconomics', 'nation', 'supply', 'because'] ...
Chosen q0 words (from the 100-term vocabulary): ['nation', 'water', 'studies']

--- Five strongest features in mu_R (relevant centroid) ---


,term,weight
0,describes,0.108498
1,representing,0.094316
2,stock,0.082597
3,global,0.079187
4,bonds,0.076749



--- Five strongest features in mu_NR (non-relevant centroid) ---


,term,weight
0,prices,0.043082
1,utility,0.041999
2,macroeconomics,0.040736
3,sum,0.040270
4,direct,0.037684



--- Five strongest features in q_opt ---


...


--- Part D: q_opt top-5 - weight vs. q0 on that same coordinate ---


,term,q_opt,q0_at_same_coord
0,nation,0.327561,0.57735
1,studies,0.282621,0.57735
2,water,0.279284,0.57735
3,describes,0.081374,0.00000
4,representing,0.070737,0.00000



The three q0 words: q0 vs. q_opt on those coordinates


,q0_word,q0_weight,q_opt_weight
0,nation,0.57735,0.327561
1,water,0.57735,0.279284
2,studies,0.57735,0.282621



Three documents closest to q_opt (cosine similarity):
  1. sim=0.4525 - Capital (economics)
  2. sim=0.4434 - Externality
  3. sim=0.3069 - Marginalism
